In [1]:
import os
import pandas as pd
import json
import numpy as np


#### Baseline

In [2]:
def get_baseline_result_df(dataset,test_file_name='test_result.json'): 
    if dataset == 'brexit':
        result_path = "../results/roberta-base/brexit/Hate/"
        n_anns = 6

    elif dataset == 'mfrc':
        result_path = "../results/roberta-base/mfrc/Moral/"
        n_anns = "24"

        
    
    records = []
    for root, subdirs, files in os.walk(result_path):
        if f'mtl_{n_anns}' in root :
            for file in files:
                    if file == f'{test_file_name}':
                        with open(os.path.join(root, file), 'r') as f:
                            data = json.load(f)
                            seed=root.split('/')[-4]
                            budget = root.split('/')[-3].split('_')[-1]
                            for i,ann in enumerate(data['task_name']):
                                records.append({'seed':seed, 'budget':budget, 'Annotator':ann, 
                                                'f1': data['f1'][i]})
    df = pd.DataFrame(records)
    return df

def get_baseline_performance_variance_across_anns(df):
    df= df.groupby(['seed', 'budget']).agg({'f1': ['mean', 'std']}).reset_index()
    df.columns = df.columns.map('_'.join).str.strip('_')
    df = df.groupby(['budget']).agg({'f1_mean': 'mean', 'f1_std': 'mean'}).reset_index()
    return df

### Our method (MTLs)

In [3]:
def get_mtl_result_df(dataset,test_file_name='test_result.json'): 
    if dataset == 'brexit':
        result_path = "../results/roberta-base/brexit/Hate/"
        n_anns = 6

    elif dataset == 'mfrc':
        result_path = "../results/roberta-base/mfrc/Moral/"
        n_anns = 24
        
    records = []
    for root, subdirs, files in os.walk(result_path):
        if (f'mtl' in root) and (f'mtl_{n_anns}' not in root) and ("few_shot") not in root :
            for file in files:
                    if file == f'{test_file_name}':
                        with open(os.path.join(root, file), 'r') as f:
                            data = json.load(f)
                            seed=root.split('/')[-4]
                            budget = root.split('/')[-3].split('_')[-1]
                            for i,ann in enumerate(data['task_name']):
                                records.append({'seed':seed, 'budget':budget, 'Annotator':ann, 
                                                'f1': data['f1'][i]})
    df = pd.DataFrame(records)
    return df

### Our method (FS)

In [4]:
def get_fs_result_df(dataset,test_file_name='test_result.json'): 
    if dataset == 'brexit':
        result_path = "../results/roberta-base/brexit/Hate/"
        n_anns = 6

    elif dataset == 'mfrc':
        result_path = "../results/roberta-base/mfrc/Moral/"
        n_anns = 24
        
    records = []
    for root, subdirs, files in os.walk(result_path):
        if "few_shot" in root :
            for file in files:
                    if file == f'{test_file_name}':
                        with open(os.path.join(root, file), 'r') as f:
                            data = json.load(f)
                            annotator = root.split('/')[-2]
                            k_shot = root.split('/')[-3]
                            sampling = root.split('/')[-5]
                            mtl = root.split('/')[-1]
                            budget = root.split("/")[-7].split('_')[-1]
                            seed = root.split('/')[-8]
                            
                            records.append({'seed': seed, 'sampling' : sampling, 'budget': budget,
                                            'k_shot': k_shot ,
                                            'Annotator':annotator,  'mtl': mtl,
                                            'f1': data['f1']})
    df = pd.DataFrame(records)
    return df

In [5]:

def get_mtl_fs_df(df, df_mtl):

    sampling_methods = df['sampling'].unique()
    shots = df['k_shot'].unique()

    dfs_sample_shot = []
    columns = ['seed', 'budget', 'Annotator', 'f1']
    for sampling in sampling_methods:
        for shot in shots:
            df_samp_shot = df[(df['sampling'] == sampling) & (df['k_shot'] == shot)]
            df_samp_shot = df_samp_shot[columns]
            df_samp_shot = pd.concat([df_samp_shot, df_mtl])
            
            # first get the avg of each annotator
            df_samp_shot = df_samp_shot.groupby(['seed', 'budget', 'Annotator']).agg({'f1': 'mean'}).reset_index()
            
            #get the average and std across annotators
            df_samp_shot = df_samp_shot.groupby(['seed', 'budget']).agg({'f1': ['mean','std']}).reset_index()
            df_samp_shot.columns = df_samp_shot.columns.map('_'.join).str.strip('_')
            
            # get the mean of f1 and mean of std of annotators across seeds
            df_samp_shot = df_samp_shot.groupby(['budget']).agg({'f1_mean': 'mean', 'f1_std': 'mean'}).reset_index()
            df_samp_shot['sampling'] = sampling
            df_samp_shot['k_shot'] = shot
            dfs_sample_shot.append(df_samp_shot)
            
    df_sample_shot = pd.concat(dfs_sample_shot)
    return df_sample_shot



### Brexit

In [64]:
dataset = 'brexit'

df_mtl= get_mtl_result_df(dataset)
df = get_fs_result_df(dataset)

df = get_mtl_fs_df(df, df_mtl)


In [65]:
df_pivot = df.pivot(index=['k_shot', 'sampling'],
                    columns='budget').sort_index()
df_pivot.columns = df_pivot.columns.map('_'.join).str.strip('_')
df_pivot = df_pivot.round(3)
# df_pivot['0.5']=  df_pivot['f1_mean_0.5'].astype(str) + "_{(" +df_pivot['f1_std_0.5'].astype(str) + ")}"
# df_pivot['0.66']=  df_pivot['f1_mean_0.66'].astype(str) + "_{(" +df_pivot['f1_std_0.66'].astype(str) + ")}"
# df_pivot['0.83'] = df_pivot['f1_mean_0.83'].astype(str) + "_{(" +df_pivot['f1_std_0.83'].astype(str) + ")}"
# df_brexit = df_pivot[['0.5', '0.66', '0.83']]
df_brexit = df_pivot[['f1_std_0.5',
       'f1_std_0.66', 'f1_std_0.83']]
df_brexit

f1_std_0.5  f1_std_0.66  f1_std_0.83
k_shot sampling                                               
128    strategy_balanced       0.093        0.108        0.117
       strategy_high_dis       0.111        0.120        0.124
       strategy_mv             0.137        0.142        0.132
       strategy_random         0.131        0.127        0.136
16     strategy_balanced       0.119        0.127        0.132
       strategy_high_dis       0.141        0.134        0.127
       strategy_mv             0.138        0.137        0.131
       strategy_random         0.137        0.139        0.135
32     strategy_balanced       0.113        0.125        0.133
       strategy_high_dis       0.145        0.138        0.131
       strategy_mv             0.140        0.137        0.129
       strategy_random         0.139        0.135        0.138
64     strategy_balanced       0.109        0.119        0.127
       strategy_high_dis       0.137        0.126        0.123
       strategy_mv             0.145        0.146        0.126
       strategy_random         0.131        0.128        0.132

### MFRC

In [66]:
dataset = 'mfrc'

columns = ['seed', 'budget', 'Annotator', 'f1']

df_mtl= get_mtl_result_df(dataset)
df = get_fs_result_df(dataset)

df = get_mtl_fs_df(df, df_mtl)

In [67]:
df_pivot = df.pivot(index=['k_shot', 'sampling'],
                    columns='budget').sort_index()
df_pivot.columns = df_pivot.columns.map('_'.join).str.strip('_')
df_pivot = df_pivot.round(3)
# df_pivot['0.25']=  df_pivot['f1_mean_0.25'].astype(str) +  "_{(" +df_pivot['f1_std_0.25'].astype(str) + ")}"
# df_pivot['0.5']=  df_pivot['f1_mean_0.5'].astype(str) +  "_{(" +df_pivot['f1_std_0.5'].astype(str) + ")}"
# df_pivot['0.75'] = df_pivot['f1_mean_0.75'].astype(str) +  "_{(" +df_pivot['f1_std_0.75'].astype(str) + ")}"

# df_mfrc = df_pivot[['0.25', '0.5', '0.75']]
df_mfrc = df_pivot[["f1_std_0.25",	"f1_std_0.5", "f1_std_0.75"]]
df_mfrc

f1_std_0.25  f1_std_0.5  f1_std_0.75
k_shot sampling                                               
128    strategy_balanced        0.117       0.122        0.121
       strategy_high_dis        0.130       0.129        0.127
       strategy_mv              0.126       0.128        0.127
       strategy_random          0.134       0.133        0.128
16     strategy_balanced        0.119       0.124        0.124
       strategy_high_dis        0.129       0.130        0.126
       strategy_mv              0.131       0.130        0.127
       strategy_random          0.137       0.135        0.130
32     strategy_balanced        0.120       0.125        0.124
       strategy_high_dis        0.131       0.130        0.126
       strategy_mv              0.130       0.130        0.127
       strategy_random          0.136       0.134        0.129
64     strategy_balanced        0.119       0.124        0.123
       strategy_high_dis        0.131       0.130        0.126
       strategy_mv              0.130       0.129        0.127
       strategy_random          0.136       0.135        0.129

In [68]:
df_merge = pd.merge(df_brexit, df_mfrc, left_index=True, right_index=True)

In [69]:
table = df_merge.to_latex(caption='Variance of performance across annotators', label='tab:variance' ,
                                 na_rep='\cellcolor{lightgray}', multirow=True, column_format="ll|lll|lll")
# header = [ "$50\%$" ,"$66\%$", "$83\%$" ,"$100\%$" ,  "$25\%$" , "$50\%$" , "$75\%$" , "$100\%$" ]

table = table.replace('_{', '\\textsubscript{')

table = table.replace('strategy_balanced', '$\mathcal{S}_{bal}$')
table = table.replace('strategy_high_dis', '$\mathcal{S}_{dis}$')
table = table.replace('strategy_mv', '$\mathcal{S}_{mv}$')
table = table.replace('strategy_random', '$\mathcal{S}_{rand}$')

table = table.replace('budget', "$\%B_f$")
table = table.replace('full_b', "MTL")

table= table.replace('\\begin{table}', '\\begin{table*}[h!]\small')
table= table.replace('\\end{table}', '\\end{table*}')

table = table.replace('toprule', 'toprule   & \multicolumn{3}{c|}{Brexit} & \multicolumn{3}{c}{mfrc} \\\\')

# table = table.replace('k-shot &  &  &  &  &  &  &  &  \\\\' , "" )

# table = table.replace('0', '\multicolumn{2}{c|}{MTL}')

# # table = table.replace('\multirow[t]', '\multirow[c]')
# $k=16$
table = table.replace('16 &', '$k=16$ &')
table = table.replace('32 &', '$k=32$ &')
table = table.replace('64 &', '$k=64$ &')
table = table.replace('128 &', '$k=128$ &')

print(table)

\begin{table*}[h!]\small
\caption{Variance of performance across annotators}
\label{tab:variance}
\begin{tabular}{ll|lll|lll}
\toprule   & \multicolumn{3}{c|}{Brexit} & \multicolumn{3}{c}{mfrc} \\
 &  & f1_std_0.5_x & f1_std_0.66 & f1_std_0.83 & f1_std_0.25 & f1_std_0.5_y & f1_std_0.75 \\
k_shot & sampling &  &  &  &  &  &  \\
\midrule
\multirow[t]{4}{*}{128} & $\mathcal{S}_{bal}$ & 0.093000 & 0.108000 & 0.117000 & 0.117000 & 0.122000 & 0.121000 \\
 & $\mathcal{S}_{dis}$ & 0.111000 & 0.120000 & 0.124000 & 0.130000 & 0.129000 & 0.127000 \\
 & $\mathcal{S}_{mv}$ & 0.137000 & 0.142000 & 0.132000 & 0.126000 & 0.128000 & 0.127000 \\
 & $\mathcal{S}_{rand}$ & 0.131000 & 0.127000 & 0.136000 & 0.134000 & 0.133000 & 0.128000 \\
\cline{1-8}
\multirow[t]{4}{*}{16} & $\mathcal{S}_{bal}$ & 0.119000 & 0.127000 & 0.132000 & 0.119000 & 0.124000 & 0.124000 \\
 & $\mathcal{S}_{dis}$ & 0.141000 & 0.134000 & 0.127000 & 0.129000 & 0.130000 & 0.126000 \\
 & $\mathcal{S}_{mv}$ & 0.138000 & 0.137000 & 0.13100

### baselines

In [70]:
dataset = 'mfrc'
df = get_baseline_result_df(dataset)
df_mean_var = get_baseline_performance_variance_across_anns(df)
df_mean_var = df_mean_var.round(4)
df_mean_var['f1']  = df_mean_var['f1_mean'].astype(str) + "(" +df_mean_var['f1_std'].astype(str) + ")"
mfrc_baeline = df_mean_var.transpose().drop(['f1_mean', 'f1_std'])

In [71]:
dataset = 'brexit'
df = get_baseline_result_df(dataset)
df_mean_var = get_baseline_performance_variance_across_anns(df)
df_mean_var = df_mean_var.round(3)
df_mean_var['f1']  = df_mean_var['f1_mean'].astype(str) + "(" +df_mean_var['f1_std'].astype(str) + ")"
brexit_baseline = df_mean_var.transpose().drop(['f1_mean', 'f1_std'])

In [72]:
# print(pd.merge(brexit_baseline, mfrc_baeline, left_index=True, right_index=True).to_latex(caption='Variance across annotators baseline ', label='tab:baseline'))





pd.merge(brexit_baseline, mfrc_baeline, left_index=True, right_index=True)


,0_x,1_x,2_x,3_x,0_y,1_y,2_y,3_y
budget,0.5,0.66,0.83,1.0,0.25,0.5,0.75,1.0
f1,0.417(0.168),0.449(0.139),0.418(0.131),0.431(0.13),0.763(0.128),0.7725(0.136),0.7723(0.1269),0.7755(0.1299)


### table of delta variance

In [73]:
print(pd.merge(brexit_baseline, mfrc_baeline, left_index=True, right_index=True).to_latex())
# baseline_var_dict = {'brexit': {'0.5': 0.168, "0.66": 0.139, "0.83": 0.131, "1.0": 0.13},
#                      'mfrc': {'0.25': 0.128, "0.5": 0.136, "0.75": 0.127 , "1.0": 0.13}}

\begin{tabular}{lllllllll}
\toprule
 & 0_x & 1_x & 2_x & 3_x & 0_y & 1_y & 2_y & 3_y \\
\midrule
budget & 0.5 & 0.66 & 0.83 & 1.0 & 0.25 & 0.5 & 0.75 & 1.0 \\
f1 & 0.417(0.168) & 0.449(0.139) & 0.418(0.131) & 0.431(0.13) & 0.763(0.128) & 0.7725(0.136) & 0.7723(0.1269) & 0.7755(0.1299) \\
\bottomrule
\end{tabular}



In [75]:
df_merge = pd.merge(df_brexit, df_mfrc, left_index=True,
                    right_index=True).round(3)
table = df_merge.to_latex(caption='Variance of performance across annotators', label='tab:variance',
                          float_format=lambda x: f'{x:.3f}' if pd.notna(
                              x) else '',
                          na_rep='\cellcolor{lightgray}', multirow=True, column_format="ll|lll|lll")
# header = [ "$50\%$" ,"$66\%$", "$83\%$" ,"$100\%$" ,  "$25\%$" , "$50\%$" , "$75\%$" , "$100\%$" ]

table = table.replace('_{', '\\textsubscript{')

table = table.replace('strategy_balanced', '$\mathcal{S}_{bal}$')
table = table.replace('strategy_high_dis', '$\mathcal{S}_{dis}$')
table = table.replace('strategy_mv', '$\mathcal{S}_{mv}$')
table = table.replace('strategy_random', '$\mathcal{S}_{rand}$')

table = table.replace('budget', "$\%B_f$")
table = table.replace('full_b', "MTL")

table = table.replace('\\begin{table}', '\\begin{table}[h!]\small')
table = table.replace('\\end{table}', '\\end{table}')

table = table.replace(
    'toprule', 'toprule   & \multicolumn{3}{c|}{Brexit} & \multicolumn{3}{c}{mfrc} \\\\')

# table = table.replace('k-shot &  &  &  &  &  &  &  &  \\\\' , "" )

# table = table.replace('0', '\multicolumn{2}{c|}{MTL}')

# # table = table.replace('\multirow[t]', '\multirow[c]')
# $k=16$
# table = table.replace('16 &', '$k=16$ &')
# table = table.replace('32 &', '$k=32$ &')
# table = table.replace('64 &', '$k=64$ &')
# table = table.replace('128 &', '$k=128$ &')

print(table)

\begin{table}[h!]\small
\caption{Variance of performance across annotators}
\label{tab:variance}
\begin{tabular}{ll|lll|lll}
\toprule   & \multicolumn{3}{c|}{Brexit} & \multicolumn{3}{c}{mfrc} \\
 &  & f1_std_0.5_x & f1_std_0.66 & f1_std_0.83 & f1_std_0.25 & f1_std_0.5_y & f1_std_0.75 \\
k_shot & sampling &  &  &  &  &  &  \\
\midrule
\multirow[t]{4}{*}{128} & $\mathcal{S}_{bal}$ & 0.093 & 0.108 & 0.117 & 0.117 & 0.122 & 0.121 \\
 & $\mathcal{S}_{dis}$ & 0.111 & 0.120 & 0.124 & 0.130 & 0.129 & 0.127 \\
 & $\mathcal{S}_{mv}$ & 0.137 & 0.142 & 0.132 & 0.126 & 0.128 & 0.127 \\
 & $\mathcal{S}_{rand}$ & 0.131 & 0.127 & 0.136 & 0.134 & 0.133 & 0.128 \\
\cline{1-8}
\multirow[t]{4}{*}{16} & $\mathcal{S}_{bal}$ & 0.119 & 0.127 & 0.132 & 0.119 & 0.124 & 0.124 \\
 & $\mathcal{S}_{dis}$ & 0.141 & 0.134 & 0.127 & 0.129 & 0.130 & 0.126 \\
 & $\mathcal{S}_{mv}$ & 0.138 & 0.137 & 0.131 & 0.131 & 0.130 & 0.127 \\
 & $\mathcal{S}_{rand}$ & 0.137 & 0.139 & 0.135 & 0.137 & 0.135 & 0.130 \\
\cline{1-8}
\m